# 03 — DistilBERT Inference

Use `distilbert-base-uncased-finetuned-sst-2-english` from HuggingFace as a zero-shot sentiment analyzer. No fine-tuning needed. Runs on CPU (~5 min for 500 samples).

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import load_imdb_data, get_sample
from src.bert_utils import get_sentiment_pipeline, batch_predict, parse_pipeline_output
from src.evaluate import compute_metrics, print_report, confusion_matrix_df

sns.set_theme(style="whitegrid")
%matplotlib inline


## Load 500 Test Samples

In [ ]:
_, test_df = load_imdb_data()
sample = get_sample(test_df, n=500)
print(sample["label"].value_counts().rename({0: "Negative", 1: "Positive"}))


## Run Inference

In [ ]:
pipe    = get_sentiment_pipeline()
results = batch_predict(sample["text"].tolist(), pipe, batch_size=32)
y_pred, y_scores = parse_pipeline_output(results)
y_true = sample["label"].values


## Metrics

In [ ]:
print_report("DistilBERT (distilbert-base-uncased-finetuned-sst-2-english)", y_true, y_pred)
metrics = compute_metrics(y_true, y_pred, y_scores)
pd.Series(metrics, name="Score").to_frame()


## Confusion Matrix

In [ ]:
cm = confusion_matrix_df(y_true, y_pred).values
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Pred Neg", "Pred Pos"],
            yticklabels=["True Neg", "True Pos"])
ax.set_title("DistilBERT — Confusion Matrix")
plt.tight_layout()
plt.savefig("../results/distilbert_confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()


## Confidence Score Distribution

In [ ]:
correct   = y_pred == y_true
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, mask, title in zip(axes, [correct, ~correct], ["Correct Predictions", "Wrong Predictions"]):
    ax.hist(y_scores[mask], bins=30, color="#7bc4e0" if mask.all() else "#e07b7b",
            edgecolor="none", alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel("Confidence Score (P[Positive])")
    ax.set_ylabel("Count")

plt.tight_layout()
plt.savefig("../results/distilbert_confidence.png", dpi=120, bbox_inches="tight")
plt.show()


## Sample Predictions

In [ ]:
for i in range(8):
    pred_label = "POSITIVE ✅" if y_pred[i] == 1 else "NEGATIVE ✅" if y_true[i] == 0 else "NEGATIVE ❌"
    true_label = "POS" if y_true[i] == 1 else "NEG"
    print(f"[{i+1}] TRUE: {true_label}  | PRED: {'POS' if y_pred[i]==1 else 'NEG'}  | CONF: {y_scores[i]:.3f}")
    print(f"     {sample['text'].iloc[i][:120]}...\n")
